In [25]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [85]:
# Inputs
checkpoint_file="../../../saved_models/ConstantPos-final.ckpt"
data_file = "../../../data/ConstPos_0709.csv"
train_perc = 80

norm_trained = True

### Setup

In [86]:
# (No edit) Imports
import os
import json

import sys
from pathlib import Path
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import ProjectedGradientDescent
from art.attacks.evasion import FastGradientMethod
from art.attacks.evasion import SaliencyMapMethod
from art.attacks.evasion import CarliniL2Method
# from art.attacks.evasion import Carlin
from sklearn.preprocessing import MinMaxScaler

sys.path.append(str(Path.cwd().parents[2]))

import os

import torch
from utils.models import OBU
from utils.functions import get_windowed_data, load_model_checkpoint

from sklearn.metrics import precision_score, recall_score, f1_score

In [87]:
# (No edit) Load models, define wrappers
model = load_model_checkpoint(checkpoint_file, gpu=False)

class SequenceCrossEntropy(nn.Module):
    def __init__(self):
        super().__init__()
        self.loss = nn.CrossEntropyLoss()

    def forward(self, a, b):
        if a.dim() == 3:
            # sequence output: (batch, seq_len, num_classes)
            if b.dim() == 3:
                b = b.argmax(dim=-1)
            return self.loss(a.permute(0, 2, 1), b.long())
        else:
            # collapsed output: (batch, num_classes)
            if b.dim() == 2:
                b = b.argmax(dim=-1)
            return self.loss(a, b.long())


class NormalizedCfCWrapper(nn.Module):
    def __init__(self, modena_model, data_min, data_max):
        super().__init__()
        self.modena_model = modena_model
        if not norm_trained:
            # MUST wrap in torch.tensor() explicitly — do not assign raw numpy arrays
            self.register_buffer("data_min", torch.tensor(data_min, dtype=torch.float32))
            self.register_buffer("data_max", torch.tensor(data_max, dtype=torch.float32))

    def forward(self, x_normalized):        
        if not norm_trained:
            x_raw = x_normalized * (self.data_max - self.data_min) + self.data_min
        else:
            x_raw = x_normalized
        logits, _ = self.modena_model(x_raw)

        return logits

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/anaconda3/envs/reu/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!


In [88]:
# (no edit) FUNCTION to set model variables
wrapped_model, criterion, optimizer, classifier = None, None, None, None
def set_model_vars (in_x_train):
    global wrapped_model, criterion, optimizer, classifier
    wrapped_model = NormalizedCfCWrapper(modena_model=model.model,
                                        data_min=in_x_train.amin(dim=(0,1)), # x_train.min(axis=(0,1)) in numpy
                                        data_max=in_x_train.amax(dim=(0,1)))
    criterion = SequenceCrossEntropy()
    optimizer = optim.Adam(
        wrapped_model.parameters(),
        lr=0.001
    )

    classifier = PyTorchClassifier(
        model=wrapped_model,
        loss=criterion,
        optimizer=optimizer,
        input_shape=(10, 8),
        nb_classes=2, # the range [0, 3]; WRONG: the number of unique classes in y_test, len(np.unique(y_test.numpy()))
        clip_values=(0.0, 1.0), # for normalized
        device_type="cpu"
    )

In [89]:
# (No edit) Load data for og model (very time consuming part)
(x_train, y_train), (x_test, y_test), fed_dataset, scaler = get_windowed_data(data_file, 
                                                                      normalize=norm_trained, 
                                                                      train_perc=train_perc)



### Check Model

In [90]:
# TEST: Test the model - ensure there's a high accuracy with the original model
set_model_vars(x_train)
print("TESTING |", f"norm_trained: {norm_trained}", "| NO wrapper")
out = model.test(x_test, y_test, mathy=True)

print(f"Out: {out}")

TESTING | norm_trained: True | NO wrapper
0.7376457251905284
0.9759886700836256
Model got 1110160/1247740 right.
Accuracy: 0.8897366438520846, Precision: 0.7376457251905284, Recall: 0.9759886700836256, F1 Score: 0.840242087001758
877040, 70.29028483498165% Zeroes, 370700 Non Zero entries.
Out: (0.840242087001758, 0.9759886700836256, 0.7376457251905284, 0.8897366438520846)


In [91]:
# TEST: Benign test
set_model_vars(x_train)

benign_y_test = y_test.numpy()

benign_predictions = classifier.predict(x_test, batch_size=64)
benign_pred_classes = np.argmax(benign_predictions, axis=-1)  # (N, seq_len)
true_flat = benign_y_test.flatten()
pred_flat = benign_pred_classes.flatten()

accuracy = np.sum(benign_pred_classes == benign_y_test) / benign_pred_classes.size
precision =  precision_score(true_flat, pred_flat, average="weighted", zero_division=0)
recall = recall_score(true_flat, pred_flat, average="weighted", zero_division=0)
f1 = f1_score(true_flat, pred_flat, average="weighted", zero_division=0)

print(f"Benign accuracy: {accuracy}")
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

Benign accuracy: 0.8897366438520846
Precision: 0.9137932399251002
Recall: 0.8897366438520846
F1: 0.8933641001180735


### Wrapper Tests

In [92]:
import time

def adv_test(end_index: int, path: str, filename: str, Attack, **kwargs):
    print(f"=== Attack: {Attack.__name__}, kwargs: {kwargs} ===")
    start = time.time_ns()

    true_flat = y_test.numpy()[:end_index].flatten()

    attack = Attack(classifier, **kwargs)
    x_test_adv = attack.generate(x=x_test.numpy()[:end_index])

    adversarial_predictions = classifier.predict(x_test_adv)
    pred_flat = np.argmax(adversarial_predictions, axis=-1).flatten()

    elapsed_ns = time.time_ns() - start

    # Confusion matrix (binary: 1=attack, 0=benign)
    TP = int(np.sum((true_flat == 1) & (pred_flat == 1)))
    TN = int(np.sum((true_flat == 0) & (pred_flat == 0)))
    FP = int(np.sum((true_flat == 0) & (pred_flat == 1)))
    FN = int(np.sum((true_flat == 1) & (pred_flat == 0)))

    total = TP + TN + FP + FN
    accuracy  = (TP + TN) / total if total > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    asr       = FN / (TP + FN) if (TP + FN) > 0 else 0.0  # false negative rate
    fpr       = FP / (FP + TN) if (FP + TN) > 0 else 0.0

    print(f"Accuracy:           {accuracy:.4f}")
    print(f"Precision:          {precision:.4f}")
    print(f"Recall:             {recall:.4f}")
    print(f"F1:                 {f1:.4f}")
    print(f"ASR (FNR):          {asr:.4f}")
    print(f"False Positive Rate:{fpr:.4f}")
    print(f"TP={TP}, TN={TN}, FP={FP}, FN={FN}")
    print(f"Time elapsed:       {elapsed_ns / 1e9:.2f}s")

    os.makedirs(path, exist_ok=True)
    metrics = {
        "endIndex": end_index,
        "attack": Attack.__name__,
        "timeElapsedSec": elapsed_ns / 1e9,
        "kwargs": kwargs,
        "metrics": {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "attackSuccessRate": asr,
            "falsePositiveRate": fpr,
            "TP": TP,
            "TN": TN,
            "FP": FP,
            "FN": FN
        },
        "files":{
            "checkpointFile": checkpoint_file,
            "dataFile": data_file
        }
    }
    output_path = f"{path}/{filename}"
    with open(output_path, "w") as f:
        json.dump(metrics, f, indent=4)
    print(f"Saved metrics to {output_path}")

    return metrics

In [93]:
# FGSM - eps = 0.1
# PGD - eps = 0.1, 
# JSMA - swept gamma | collapsed_output = true, theta = amount perturbed, gamma = fraction of features perturbed

for i in range(1, 31):
    eps = float(i/100)
    adv_test(end_index = len(y_test.numpy()),
             path = "../final_data/constantpos/fgsm",
             filename=f"adv_eps_{eps}.json",
             Attack=FastGradientMethod,
             eps=eps
             )

=== Attack: FastGradientMethod, kwargs: {'eps': 0.01} ===
Accuracy:           0.8268
Precision:          0.6369
Recall:             0.9700
F1:                 0.7689
ASR (FNR):          0.0300
False Positive Rate:0.2337
TP=359565, TN=672045, FP=204995, FN=11135
Time elapsed:       19.25s
Saved metrics to ../final_data/constantpos/fgsm/adv_eps_0.01.json
=== Attack: FastGradientMethod, kwargs: {'eps': 0.02} ===
Accuracy:           0.8065
Precision:          0.6099
Recall:             0.9682
F1:                 0.7484
ASR (FNR):          0.0318
False Positive Rate:0.2618
TP=358916, TN=647439, FP=229601, FN=11784
Time elapsed:       19.38s
Saved metrics to ../final_data/constantpos/fgsm/adv_eps_0.02.json
=== Attack: FastGradientMethod, kwargs: {'eps': 0.03} ===
Accuracy:           0.7990
Precision:          0.6008
Recall:             0.9643
F1:                 0.7403
ASR (FNR):          0.0357
False Positive Rate:0.2708
TP=357467, TN=639520, FP=237520, FN=13233
Time elapsed:       19.36s
S

In [ ]:
for i in range(1, 31):
    eps = float(i/100)
    adv_test(end_index = len(y_test.numpy()),
             path = "../final_data/fgsm",
             filename=f"adv_eps_{eps}.json",
             Attack=ProjectedGradientDescent,
             eps=eps
             )